# 1D CNN Gesture Classifier

## Overview

This notebook implements a **1D Convolutional Neural Network (CNN)** for dynamic hand-gesture recognition using data-glove time-series recordings.

---

### From 2D to 1D Convolution — Connecting to Lecture 10

Your COMP3308 lecture (Lecture 10) introduced CNNs on **2D images**: a small filter slides across both the width and height of the image, detecting local patterns like edges or curves.  
Here we use a **1D CNN**, where the same idea is applied to **time** instead of space.

| Concept | 2D CNN (images, from lecture) | 1D CNN (this notebook) |
|---|---|---|
| Input | (height, width, colour_channels) | (timesteps, sensor_channels) |
| Filter slides over | spatial x & y | time axis only |
| Filter detects | spatial feature (edge, curve) | temporal pattern (finger flick, wrist rotation) |
| Feature map | 2D activation grid | 1D activation sequence |

Concretely, given an input tensor of shape `(batch, T, C)` — where `T` is the number of time steps and `C` is the number of sensor channels — a `Conv1D` layer with `kernel_size=k` computes dot products over windows of `k` **consecutive time steps** across all `C` channels simultaneously.

**Example:** With `kernel_size=5` and `C=180` channels, each filter looks at a 5-sample window (≈160 ms at 31 Hz) across ALL 180 sensor readings at once and produces one scalar output — exactly like a 2D filter that is 5×180 in size.

---

### Why 1D CNN for Glove Data?

CNNs are highly effective for multi-channel wearable sensor data:
- **Local temporal patterns** (e.g. a rapid finger extension) are captured by small kernels.
- **Translational invariance** along time — the same wrist-flexion motion is detected regardless of where in the 5-second window it occurs.
- Trains significantly **faster** than recurrent networks (LSTM/GRU) because convolutions are fully parallelisable.

Compared to the LSTM notebook (`01_LSTM.ipynb`), 1D CNNs require less data to converge for fixed-length windows.

---

### Data Schema

Column naming convention: `{hand}_{segment}_{loc}_{channel}`  
e.g. `left_thumb_mid_ax`, `right_index_prox_pitch`, `left_thumb_mcp_flex`

| Sensor group | Columns |
|---|---|
| Finger distal IMU (`mid`) | `ax, ay, az, pitch, roll, yaw` per finger × hand |
| Finger proximal IMU (`prox`) | `ax, ay, az, pitch, roll, yaw` per finger × hand |
| Flex sensors | `mcp_flex, pip_flex` per finger × hand |
| Palm IMU (`palm_prox`) | `ax, ay, az, yaw, pitch, roll` per hand |
| Wrist IMU | `ax, ay, az, heading, pitch, roll` per hand |
| **EXCLUDED** | `palm_mid_*` (always 0) and `palm_flex` (always -1) |

Sampling rate: **~31 Hz** (~156 rows per 5-second trial).

---

### Pipeline at a Glance

```
Install → Imports → CONFIG → Build feature columns → Load dataset
→ Train/Val/Test split → Define CNN model → Train → Evaluate → Save → Saliency
```

---

## Cell 1 — Install Dependencies

Run this cell once before running anything else.  
It upgrades the required libraries to compatible versions in the current Python environment.  
> **Note:** If the kernel was freshly started you may need to restart it after installation and then re-run all cells.

In [ ]:
"""
Install / upgrade all Python packages required by this notebook.

Why each package is needed:
  tensorflow  — the deep-learning framework; provides Keras layers (Conv1D,
                Dense, etc.), the training loop, and automatic differentiation
                (used in the saliency cell).
  numpy       — fast array operations; all data is stored as numpy arrays
                before being fed into TensorFlow.
  pandas      — read CSV files and manipulate tabular data.
  scikit-learn — train/val/test split, classification_report, confusion_matrix.
  scipy       — optional signal filtering (Butterworth LP, moving average, etc.)
                applied to the raw sensor streams before training.
  matplotlib  — plotting learning curves, confusion matrices, saliency maps.
  seaborn     — higher-level heatmap / confusion matrix visualisation.
"""
import sys, subprocess

pkgs = [
    "tensorflow",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "seaborn",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + pkgs,
    check=True,
)
print("All dependencies ready.")


---

## Cell 2 — Imports

Import every library used in the notebook.  
Keeping all imports in one cell makes dependency problems easy to spot immediately.  
The `utils.data_loader` module (in the parent directory) handles all CSV loading, filtering, resampling, and normalisation.

In [ ]:
"""
Central imports cell — all external libraries and project utilities are loaded here.

After this cell runs you have access to:
  tf / keras / layers / Model  — TensorFlow / Keras deep-learning API
  np, pd                       — numerical and tabular data manipulation
  plt, sns                     — plotting
  Path                         — convenient file-path handling (pathlib)
  datetime                     — timestamp for saved results
  json                         — serialise results to disk
  classification_report,
  confusion_matrix             — scikit-learn evaluation metrics
  to_categorical               — one-hot encoding (Keras utility)
  callbacks                    — EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
  build_dataset,
  split_dataset                — project-specific data pipeline (utils/)
"""
import os, sys, json, datetime
from pathlib import Path

# ── Numerical / data science ──────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras import callbacks
from tensorflow.keras.utils import to_categorical

# ── Scikit-learn evaluation metrics ───────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

# ── Project data-loading utilities ───────────────────────────────────────────
# Adds the parent directory (ML/) to Python's module search path so that
# 'utils.data_loader' can be imported regardless of the current working dir.
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
from utils.data_loader import (
    build_dataset,     # loads all CSVs, filters, resamples, normalises → (X, y)
    split_dataset,     # stratified 3-way split → (X_train, y_train), ...
    build_column_groups,
    HANDS, FINGERS, LOCS,
    IMU_ALL_CHANNELS, FLEX_CHANNELS, WRIST_ALL_CHANNELS,
)

# ── Reproducibility ───────────────────────────────────────────────────────────
# Setting seeds makes training results reproducible across runs.
# Without this, random weight initialisation and mini-batch shuffling produce
# slightly different results every time.
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print("Imports complete.")


---

## Cell 3 — Experiment Configuration

**Edit this cell to control every aspect of the pipeline.**  
Every tunable parameter lives here — no other cell needs to be edited for a standard experiment.  
This single-cell design makes it easy to track which hyperparameters produced which result.

In [ ]:
"""
Central configuration cell.

Design principle: ALL experiment parameters live here so that you can
reproduce any result by saving this cell.  No magic numbers are scattered
through the notebook.

HOW TO RUN A NEW EXPERIMENT
----------------------------
1. Change any value(s) below.
2. Kernel → Restart & Run All.
3. Compare the saved results JSON / model file with previous runs.

PARAMETER GROUPS
-----------------
DATA          — where to find the raw CSVs; sampling rate; window length
SENSOR FLAGS  — which sensor groups to include as input features
PREPROCESSING — optional temporal filter; normalisation strategy
SPLIT         — train / val / test proportions
MODEL         — CNN architecture hyperparameters
TRAINING      — learning rate, batch size, stopping criteria
"""

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 31.0       # Confirmed sampling rate (~156 rows / 5 s ≈ 31.2 Hz).
                        # Used only for display / receptive-field calculations
                        # — the loader resamples every trial to TARGET_LEN rows.
TARGET_LEN = 156        # Native timesteps per trial (5 s × ~31 Hz = ~156).
                        # Reduce (e.g. 78) to halve temporal resolution and
                        # speed up training at the cost of some detail.

# ── COLUMN / SENSOR SELECTION ─────────────────────────────────────────────────
# Toggle entire sensor groups on / off.
# All combinations produce a valid feature list — build_column_groups()
# handles the column-name arithmetic automatically.

USE_LEFT_HAND  = True    # Include all left-hand sensor columns
USE_RIGHT_HAND = True    # Include all right-hand sensor columns

# Per-finger IMU locations:
#   'mid'  = distal phalanx (fingertip segment)
#   'prox' = proximal phalanx (base segment, closest to palm)
USE_DISTAL_IMU   = True  # {hand}_{finger}_mid_{ax/ay/az/yaw/pitch/roll}
USE_PROXIMAL_IMU = True  # {hand}_{finger}_prox_{ax/ay/az/yaw/pitch/roll}

# IMU channel types — applies to finger IMUs, palm_prox IMU, and wrist IMU:
USE_ACCELEROMETER = True  # ax, ay, az   — linear acceleration (in g)
USE_ORIENTATION   = True  # yaw, pitch, roll — Euler angles (degrees)

# Flex sensors measure how much each finger joint is bent:
#   MCP = metacarpophalangeal joint (the large knuckle)
#   PIP = proximal interphalangeal joint (the middle joint)
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                          # NOTE: palm flex channels are always -1 → auto-excluded

# Palm IMU (physically located on the back of the hand — 'palm_prox'):
USE_PALM_IMU  = True      # {hand}_palm_prox_{ax/ay/az/yaw/pitch/roll}

# Wrist IMU:
USE_WRIST_IMU = True      # {hand}_wrist_{ax/ay/az/heading/pitch/roll}
                          # Note: wrist uses 'heading' instead of 'yaw'.

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# The raw CSVs have already been Butterworth LP-filtered at 5 Hz by the glove
# firmware — set FILTER_TYPE = 'none' to use them as-is.
# An additional filter can be applied here if desired.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'none'

# Normalisation converts raw sensor values into a consistent numeric range.
# This is important because different sensors (e.g. flex vs accelerometer)
# have very different physical scales.
#
# 'minmax'  → scale each channel to [0, 1]   (preserves shape, sensitive to outliers)
# 'zscore'  → zero mean, unit variance       (makes all channels equally scaled)
# 'none'    → use raw values                 (not recommended — may cause exploding grads)
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True  = normalise each 5-second trial independently.
                         #         Recommended: removes between-trial amplitude variation.
                         # False = normalise across the entire dataset (global statistics).

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
# Stratified split: each class appears in the same proportion in every subset.
TRAIN_RATIO = 0.70   # 70 % → ~280 samples for weight updates
VAL_RATIO   = 0.10   # 10 % →  ~40 samples for hyperparameter tuning / early stopping
                     # 20 % →  ~80 samples held out for final unbiased evaluation (auto)
RANDOM_SEED = 42     # Fixed seed → reproducible splits across runs

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# The CNN is built from stacked "conv blocks".  Each entry = (filters, kernel_size).
#
# filters     = number of feature detectors in that layer.
#               More filters → richer feature vocabulary; higher memory/compute cost.
# kernel_size = how many consecutive time steps each filter spans.
#               Smaller = detects fine local patterns; larger = coarser patterns.
#
# This default stack:
#   Block 1: 64 filters, kernel=7  → detects broad 7-timestep (~225 ms) patterns
#   Block 2: 128 filters, kernel=5 → detects medium patterns in the compressed sequence
#   Block 3: 256 filters, kernel=3 → detects fine patterns at the deepest level
CONV_BLOCKS    = [(64, 7), (128, 5), (256, 3)]

USE_BATCH_NORM = True   # Batch normalisation after each conv layer.
                        # Keeps activations in a healthy range → faster, more stable training.
USE_MAX_POOL   = True   # MaxPooling1D after each conv block.
                        # Halves the time dimension → doubles the receptive field.
POOL_SIZE      = 2      # Pool window size (2 = halve the sequence length each time)
DROPOUT_RATE   = 0.4    # Fraction of neurons randomly zeroed during each training step.
                        # Acts as regularisation to combat overfitting.
                        # 0.4 = 40 % of activations are dropped.
DENSE_UNITS    = [128, 64]  # Size of each fully-connected layer after global pooling.
GLOBAL_POOL    = 'average'  # How to collapse the final sequence into a fixed vector.
                            # 'average' = mean over time (soft, stable)
                            # 'max'     = strongest activation over time (sharp, fast)

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80    # Maximum number of passes over the training data.
                            # EarlyStopping will typically stop before this.
BATCH_SIZE          = 32    # Number of samples per gradient update.
                            # Larger batches = more stable gradients but more memory.
LEARNING_RATE       = 1e-3  # Initial step size for Adam optimiser.
                            # Adam adapts per-parameter learning rates automatically.
EARLY_STOP_PATIENCE = 15    # Stop training if val_loss doesn't improve for this many epochs.
                            # Prevents overfitting to the training set.

# ── DERIVED PATH CONSTANTS ────────────────────────────────────────────────────
# These are computed from the config above — do NOT edit directly.
MODEL_SAVE_PATH   = 'saved_models/02_cnn_best.keras'
RESULTS_SAVE_PATH = 'results/02_cnn_results.json'

print("Configuration loaded.")
print(f"  Target sequence length : {TARGET_LEN} timesteps  (~{TARGET_LEN/FS_HZ:.1f} s at {FS_HZ} Hz)")
print(f"  CNN blocks             : {CONV_BLOCKS}")
print(f"  Dense units            : {DENSE_UNITS}")
print(f"  Batch norm / MaxPool   : {USE_BATCH_NORM} / {USE_MAX_POOL}")
print(f"  Dropout rate           : {DROPOUT_RATE}")
print(f"  Training               : up to {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}")


---

## Cell 4 — Build Feature Column List from Config

Translates the `USE_*` boolean flags from CONFIG into an explicit list of CSV column names.  
`build_column_groups()` applies the naming convention `{hand}_{finger}_{loc}_{channel}` and automatically excludes always-zero columns.  
The number of selected columns becomes `n_features` — the depth dimension of every Conv1D layer's input.

In [ ]:
"""
Translate CONFIG flags → concrete list of CSV column names.

WHY THIS STEP EXISTS
--------------------
The raw CSVs contain 180 feature columns per trial (90 per hand).  Not all are
useful for every experiment.  For example, you might want to test the model using
only flex sensors (no IMU) to see if bend angle alone is discriminative.

build_column_groups() encodes the naming convention so you never have to manually
type column names.  Changing a USE_* flag above and re-running from here is all
that's needed.

OUTPUT
------
feature_cols : list[str]  — the exact column names to pass to build_dataset().
                            Length = n_features = the 'C' dimension of the CNN input.
"""

# Resolve hand selection: build list of ['left'], ['right'], or ['left','right']
_hands = (
    [h for h in HANDS if (USE_LEFT_HAND and h == "left")
                       or (USE_RIGHT_HAND and h == "right")]
    or HANDS  # fallback: use both if both flags are False (shouldn't happen)
)

# All five fingers are always used — individual finger toggling is not
# implemented since gesture identity depends on the pattern ACROSS all fingers.
_fingers = FINGERS

# Resolve IMU location selection: 'mid' (distal) and/or 'prox' (proximal)
_locs = (
    [l for l in LOCS
     if (USE_DISTAL_IMU   and l == "mid")
     or (USE_PROXIMAL_IMU and l == "prox")]
    or LOCS   # fallback: use both if both are False
)

# build_column_groups returns the full list of valid column names matching the
# sensor/channel combination requested.  It automatically excludes:
#   - palm_mid_*   (always-zero — no physical sensor)
#   - palm_*_flex  (always -1   — no physical sensor)
feature_cols = build_column_groups(
    hands             = _hands,
    fingers           = _fingers,
    locs              = _locs,
    include_flex      = USE_FLEX_SENSORS,
    include_palm_prox = USE_PALM_IMU,
    include_wrist     = USE_WRIST_IMU,
    include_accel     = USE_ACCELEROMETER,
    include_orient    = USE_ORIENTATION,
)

print(f"Feature columns selected: {len(feature_cols)}")
print()

# Print a grouped summary (up to 12 groups) so you can verify the selection.
# Grouping key: first 3 underscore-separated parts of the column name
# e.g. 'left_thumb_mid' groups all channels for left thumb distal IMU.
groups = {}
for c in feature_cols:
    parts = c.split("_")
    grp   = "_".join(parts[:3]) if len(parts) >= 4 else "_".join(parts[:2])
    groups.setdefault(grp, []).append(c)

for grp, cols in list(groups.items())[:12]:
    # Show the suffix (channel type) for each column in the group
    print(f"  {grp:42s} ({len(cols):2d} cols): {[c.split('_')[-1] for c in cols]}")
if len(groups) > 12:
    print(f"  ... and {len(groups)-12} more groups")


---

## Cell 5 — Load and Preprocess Dataset

`build_dataset()` walks the four gesture subfolders, loads every CSV trial, applies the optional filter, resamples each trial to `TARGET_LEN` time steps, and stacks everything into a 3D array `X` of shape `(N, TARGET_LEN, n_features)`.  
Integer class labels are stored in `y` of shape `(N,)`.  
A bar chart shows the class distribution to confirm balanced sampling.

In [ ]:
"""
Load all gesture trials from disk and build the (X, y) dataset arrays.

WHAT build_dataset DOES INTERNALLY
------------------------------------
1. discover_dataset()  — scans DATA_ROOT for subfolders; each subfolder name
                          becomes a class label (e.g. 'TwoHand_L_Fist_R_Fist...').
2. For each CSV file:
    a. load_csv()        — read the file, keep only feature_cols columns.
    b. apply_filter()    — optional temporal smoothing (FILTER_TYPE).
    c. resample()        — interpolate to exactly TARGET_LEN rows.
3. np.stack()           — combine all trials into shape (N, TARGET_LEN, C).
4. normalize()          — scale values to [0,1] or z-score (NORMALIZATION).

SHAPES AFTER THIS CELL
------------------------
X           : (N, TARGET_LEN, n_features)  — float32 input tensor
y           : (N,)                          — int64 class indices 0..3
CLASS_NAMES : ['Fist', 'Flat', 'Okay', 'Rock']   (derived from folder names)
N_FEATURES  : int  — == len(feature_cols)  == 'C' dimension of Conv1D layers
N_CLASSES   : int  — 4
"""

X, y, CLASS_NAMES, FEATURE_COLS_USED = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = feature_cols,    # the list built in Cell 4
    filter_type     = FILTER_TYPE,     # 'none' = use pre-filtered CSVs as-is
    fs              = FS_HZ,
    target_len      = TARGET_LEN,      # resample every trial to this length
    normalization   = NORMALIZATION,   # 'minmax' → values in [0, 1]
    per_sample_norm = PER_SAMPLE_NORM, # normalise each trial independently
)

# Derived constants used by later cells
N_FEATURES = X.shape[2]   # number of sensor channels (C)
N_CLASSES  = len(CLASS_NAMES)

print(f"\nDataset shapes:")
print(f"  X : {X.shape}  (samples × timesteps × features)")
print(f"  y : {y.shape}  (integer class labels)")
print(f"\nClasses ({N_CLASSES}): {CLASS_NAMES}")

# ── Class distribution bar chart ──────────────────────────────────────────────
# In supervised learning it is critical to check that classes are balanced.
# If one class has far more samples than others, the model will learn to predict
# that class more often — this is called class imbalance.
counts = [(cls, int((y == i).sum())) for i, cls in enumerate(CLASS_NAMES)]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar([c[0] for c in counts], [c[1] for c in counts],
              color=['#20808D', '#A84B2F', '#5E9E6E', '#7B5EA7'],
              edgecolor='white', linewidth=1.2)

# Annotate each bar with its count
for bar, (_, count) in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Class distribution — gesture samples', fontweight='bold', pad=10)
ax.set_xlabel('Gesture class')
ax.set_ylabel('Number of trials')
ax.set_ylim(0, max(c[1] for c in counts) * 1.15)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/02_cnn_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nClass distribution chart saved → results/02_cnn_class_distribution.png")


---

## Cell 6 — Stratified Train / Val / Test Split

Split the dataset into three non-overlapping subsets using stratified sampling — each class appears in the same proportion in every subset.

| Subset | Purpose | Proportion |
|---|---|---|
| **Train** | Gradient descent weight updates | 70 % (~280 samples) |
| **Val** | Hyperparameter tuning, early stopping | 10 % (~40 samples) |
| **Test** | Final unbiased accuracy estimate | 20 % (~80 samples) |

Labels are then one-hot encoded: integer `2` → `[0, 0, 1, 0]` for `categorical_crossentropy`.

In [ ]:
"""
Stratified three-way split → train, val, test.

WHY STRATIFIED?
---------------
With only 100 samples per class, a purely random split could accidentally put
all samples of one class into the test set.  Stratification guarantees the
class ratio is preserved in every subset — vital for reliable evaluation.

ONE-HOT ENCODING
----------------
Keras's categorical_crossentropy loss requires the target to be a probability
vector, not a single integer.  to_categorical converts:
    integer label 2  →  [0.0, 0.0, 1.0, 0.0]   (for 4 classes)

This matches the softmax output of the network, which also produces a
vector of 4 probabilities summing to 1.
"""
# Create output directories for saving models and results
Path('saved_models').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)

# split_dataset returns ((X_train, y_train), (X_val, y_val), (X_test, y_test))
# y arrays are integer labels at this point
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,   # fixed seed for reproducible splits
)

# Convert integer class labels to one-hot vectors for categorical_crossentropy
y_train_oh = to_categorical(y_train, N_CLASSES)  # shape (280, 4)
y_val_oh   = to_categorical(y_val,   N_CLASSES)  # shape ( 40, 4)
y_test_oh  = to_categorical(y_test,  N_CLASSES)  # shape ( 80, 4)

print("Shapes after split:")
print(f"  X_train : {X_train.shape}   y_train (int): {y_train.shape}  y_train_oh: {y_train_oh.shape}")
print(f"  X_val   : {X_val.shape}   y_val   (int): {y_val.shape}  y_val_oh  : {y_val_oh.shape}")
print(f"  X_test  : {X_test.shape}   y_test  (int): {y_test.shape}  y_test_oh : {y_test_oh.shape}")

# Verify class balance in each split
print("\nClass counts per split:")
for name, y_int in [("train", y_train), ("val", y_val), ("test", y_test)]:
    dist = [int((y_int == i).sum()) for i in range(N_CLASSES)]
    print(f"  {name:6s}: {dict(zip(CLASS_NAMES, dist))}")


---

## Cell 7 — Define the 1D CNN Model

The `build_cnn_model()` function constructs the network from the `CONV_BLOCKS` configuration.

### Architecture walkthrough (connecting to Lecture 10 concepts)

**Input** → shape `(batch, 156, 180)` — 156 timesteps × 180 sensor channels.

**Conv Block × 3:**  Each block contains:
1. `Conv1D(filters, kernel_size)` — applies `filters` temporal feature detectors, each spanning `kernel_size` consecutive time steps across all channels simultaneously. Think of each filter as "looking for a specific temporal shape" — just as Lecture 10's edge-detection filter looks for a specific spatial pattern.
2. `BatchNormalization` — normalises activations within the batch to keep them in a healthy range, enabling higher learning rates. Analogous to why we normalise the input features (min-max / z-score) — but applied after each layer.
3. `ReLU` activation — introduces non-linearity. Identical to the ReLU you know from fully connected networks: `max(0, x)`.
4. `MaxPooling1D(pool_size=2)` — takes the max of every 2 consecutive timesteps, halving the time dimension. Identical in spirit to the 2D max pooling in Lecture 10 but applied along one axis only.
5. `Dropout(0.4)` — randomly zeros 40% of activations during training to prevent overfitting.

**GlobalAveragePooling1D** — collapses the remaining time dimension by averaging: "on average, how strongly did each filter fire over the whole gesture?" Outputs a fixed-size vector of length `n_filters` regardless of sequence length.

**Dense FC layers + Softmax** — standard classification head identical to any fully-connected network from your lectures.

In [ ]:
"""
Build and instantiate the 1D CNN model.

CONNECTING TO LECTURE 10
-------------------------
Your lecture shows: "A filter slides over a 2D image and produces a 2D feature map."
Here:          "A 1D filter slides over the TIME axis and produces a 1D feature map."

The math is identical — it is just a dot product between the filter weights and
a window of the input — the difference is only the number of dimensions.

RECEPTIVE FIELD INTUITION
--------------------------
After the first Conv1D(kernel=7): each output unit 'sees' 7 original timesteps.
After MaxPool(2):                  stride doubles — next conv sees 2 timesteps per step.
After Conv1D(kernel=5) on pooled:  RF = 7 + (5-1)*2 = 15 original timesteps.
After MaxPool(2) again:            stride = 4.
After Conv1D(kernel=3) on pooled:  RF = 15 + (3-1)*4 = 23 original timesteps.

At 31 Hz, RF=23 timesteps ≈ 740 ms — the model can detect gesture features
spanning nearly 3/4 of a second after just 3 conv blocks.
"""

def build_cnn_model(
    seq_len,
    n_features,
    n_classes,
    conv_blocks,
    use_batch_norm = True,
    use_max_pool   = True,
    pool_size      = 2,
    dropout_rate   = 0.4,
    dense_units    = None,
    global_pool    = 'average',
):
    """
    Build a configurable 1D CNN for sequence classification.

    Parameters
    ----------
    seq_len        : int   — number of time steps (TARGET_LEN, e.g. 156)
    n_features     : int   — number of sensor channels (C, e.g. 180)
    n_classes      : int   — number of output gesture classes (e.g. 4)
    conv_blocks    : list  — [(filters, kernel_size), ...] one tuple per block
    use_batch_norm : bool  — add BatchNormalization after each Conv1D
    use_max_pool   : bool  — add MaxPooling1D after each conv block
    pool_size      : int   — pool window size (2 = halve the time dimension)
    dropout_rate   : float — fraction of units to drop during training
    dense_units    : list  — [units, ...] for each Dense layer before softmax
    global_pool    : str   — 'average' or 'max'

    Returns
    -------
    model : tf.keras.Model — compiled-ready model
    """
    if dense_units is None:
        dense_units = [128]

    # ── Input layer ────────────────────────────────────────────────────────────
    # Keras requires an explicit Input layer to define the tensor shape.
    # Shape = (seq_len, n_features) — batch dimension is implicit.
    inputs = keras.Input(shape=(seq_len, n_features), name='sensor_input')
    x = inputs

    # ── Convolutional blocks ───────────────────────────────────────────────────
    for block_idx, (filters, kernel_size) in enumerate(conv_blocks, start=1):

        # ── Conv1D ─────────────────────────────────────────────────────────────
        # filters     = how many independent feature detectors to learn.
        #               Each filter produces one output channel (feature map).
        #               With 64 filters → 64 different temporal patterns detected
        #               simultaneously.
        # kernel_size = how many consecutive time steps each filter spans.
        #               kernel=7 at 31 Hz → each filter sees ~225 ms of signal.
        # padding='same' → output has the same time length as input (zero-padded
        #                  at the edges).  Use 'causal' for real-time streaming.
        # use_bias=not use_batch_norm → bias is redundant when BatchNorm follows
        #   (BatchNorm has its own bias parameter β that serves the same purpose).
        x = layers.Conv1D(
            filters     = filters,
            kernel_size = kernel_size,
            padding     = 'same',
            use_bias    = not use_batch_norm,
            name        = f'conv_b{block_idx}_conv',
        )(x)

        if use_batch_norm:
            # Batch Normalisation: for each channel, subtract the batch mean and
            # divide by the batch standard deviation, then apply learnable scale
            # (γ) and shift (β) parameters.
            #
            # WHY IT HELPS: keeps activations from growing or shrinking across
            # layers, avoiding vanishing/exploding gradients.  You can use a
            # higher learning rate and train faster.  Think of it like doing
            # z-score normalisation after every conv layer — but adaptive.
            x = layers.BatchNormalization(name=f'conv_b{block_idx}_bn')(x)

        # ReLU activation: max(0, x)
        # Sets negative activations to zero — introduces the non-linearity that
        # allows the network to learn complex, non-linear decision boundaries.
        # Same ReLU function you know from fully-connected networks.
        x = layers.Activation('relu', name=f'conv_b{block_idx}_relu')(x)

        if use_max_pool:
            # MaxPooling1D: slide a window of size `pool_size` along the time axis
            # and keep only the MAXIMUM value in each window.
            #
            # EFFECT 1 — Dimension reduction:
            #   Input shape (T, C) → output shape (T // pool_size, C).
            #   e.g. (156, 64) → (78, 64) after the first block.
            #
            # EFFECT 2 — Translational invariance:
            #   A finger-flick pattern detected at timestep 22 or 23 produces
            #   the same pooled output → the model is less sensitive to exact timing.
            #
            # EFFECT 3 — Increased receptive field:
            #   The next Conv1D layer operates on the compressed sequence, so it
            #   effectively sees more original timesteps per filter application.
            x = layers.MaxPooling1D(
                pool_size = pool_size,
                name      = f'conv_b{block_idx}_pool',
            )(x)

        # Dropout: randomly set `dropout_rate` fraction of activations to zero
        # during each training forward pass.  At inference time, all units are
        # active (scaled by 1-dropout_rate).
        # Prevents co-adaptation of neurons → acts as an ensemble of sub-networks.
        x = layers.Dropout(dropout_rate, name=f'conv_b{block_idx}_drop')(x)

    # ── Global pooling — collapse the time dimension ──────────────────────────
    # After 3 Conv+Pool blocks with pool_size=2, a 156-timestep input becomes
    # 156 // 2 // 2 // 2 = 19 timesteps (plus edge effects from 'same' padding).
    # We still have a SEQUENCE of feature vectors — we need ONE vector for the
    # classification head.
    #
    # GlobalAveragePooling1D computes the MEAN over the remaining time steps:
    #   Input:  (batch, T', filters)   e.g. (32, 19, 256)
    #   Output: (batch, filters)            (32, 256)
    #
    # Interpretation: "on average, across the whole gesture window, how strongly
    # did filter k fire?"  — a kind of soft attention over time.
    #
    # GlobalMaxPooling1D instead takes the MAXIMUM — "at what point did filter k
    # fire most strongly?"  Choose 'average' for most cases; 'max' can work better
    # if a single sharp spike in the signal is the key discriminative feature.
    if global_pool == 'average':
        x = layers.GlobalAveragePooling1D(name='global_avg_pool')(x)
    elif global_pool == 'max':
        x = layers.GlobalMaxPooling1D(name='global_max_pool')(x)
    else:
        raise ValueError(f"global_pool must be 'average' or 'max', got '{global_pool}'")

    # ── Fully-connected classification head ───────────────────────────────────
    # After global pooling we have a plain feature vector (batch, n_filters).
    # The Dense layers are exactly like the feedforward layers you know from
    # COMP3308: weighted sum → (optional BN) → ReLU → dropout.
    for fc_idx, units in enumerate(dense_units, start=1):
        x = layers.Dense(
            units,
            use_bias = not use_batch_norm,  # BN provides its own bias
            name     = f'fc{fc_idx}',
        )(x)
        if use_batch_norm:
            x = layers.BatchNormalization(name=f'fc{fc_idx}_bn')(x)
        x = layers.Activation('relu', name=f'fc{fc_idx}_relu')(x)
        x = layers.Dropout(dropout_rate, name=f'fc{fc_idx}_drop')(x)

    # ── Output layer ──────────────────────────────────────────────────────────
    # Dense(n_classes) → one logit per class
    # Softmax converts logits to a probability distribution summing to 1.
    # e.g. [0.02, 0.05, 0.88, 0.05] → predicted class = 2 (Okay)
    # This is the same softmax you know from multi-class classification.
    outputs = layers.Dense(
        n_classes,
        activation = 'softmax',
        name       = 'output',
    )(x)

    model = Model(inputs=inputs, outputs=outputs, name='1D_CNN_Gesture')
    return model


# ── Instantiate and compile the model ─────────────────────────────────────────
model = build_cnn_model(
    seq_len        = TARGET_LEN,      # 156 time steps
    n_features     = N_FEATURES,      # e.g. 180 sensor channels
    n_classes      = N_CLASSES,       # 4 gesture classes
    conv_blocks    = CONV_BLOCKS,     # [(64,7),(128,5),(256,3)]
    use_batch_norm = USE_BATCH_NORM,
    use_max_pool   = USE_MAX_POOL,
    pool_size      = POOL_SIZE,
    dropout_rate   = DROPOUT_RATE,
    dense_units    = DENSE_UNITS,
    global_pool    = GLOBAL_POOL,
)

model.compile(
    # Adam adapts the learning rate individually for each weight based on the
    # first and second moments of the gradient — generally converges faster
    # than plain SGD for deep networks.
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    # categorical_crossentropy is the standard loss for multi-class classification
    # with one-hot encoded targets.  It measures how different the predicted
    # probability distribution is from the true one-hot distribution.
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy'],  # track accuracy alongside loss during training
)

# Print the full layer-by-layer summary including output shapes and param counts
model.summary()

# ── Receptive field analysis ──────────────────────────────────────────────────
# The receptive field (RF) is the number of original time steps that each
# output unit ultimately depends on.  It grows with each conv + pool block.
# Larger RF → the model can detect longer-range temporal patterns.
print("\n" + "─" * 58)
print("Receptive Field Analysis")
print("─" * 58)
rf         = 1     # RF before any conv (each unit sees exactly 1 timestep)
stride_acc = 1     # accumulated stride from pooling layers so far

for block_idx, (_, k) in enumerate(CONV_BLOCKS, start=1):
    # After a Conv1D with kernel k and effective stride stride_acc:
    #   RF grows by (k - 1) * stride_acc
    rf += (k - 1) * stride_acc
    print(f"  After Conv Block {block_idx} (kernel={k}): RF = {rf} time steps"
          f"  (~{rf / FS_HZ * 1000:.0f} ms)")
    if USE_MAX_POOL:
        stride_acc *= POOL_SIZE
        print(f"  After MaxPool ({POOL_SIZE}):            stride_acc = {stride_acc}  "
              f"(next conv sees {stride_acc}× more original steps)")

print(f"\nFinal receptive field  : {rf} timesteps  (~{rf / FS_HZ * 1000:.0f} ms)")
print(f"Total sequence length  : {TARGET_LEN} timesteps  (~{TARGET_LEN / FS_HZ * 1000:.0f} ms)")
print(f"Coverage fraction      : {rf / TARGET_LEN * 100:.1f}% of the window")
print("─" * 58)


---

## Cell 8 — Training with Callbacks and Learning Curves

`model.fit()` runs gradient descent for up to `EPOCHS` epochs.  Three callbacks regulate training:

| Callback | Purpose |
|---|---|
| `EarlyStopping` | Monitors `val_loss`; stops training and restores the best weights when no improvement for `EARLY_STOP_PATIENCE` epochs. Prevents overfitting. |
| `ReduceLROnPlateau` | Halves the learning rate when `val_loss` stalls. Helps the optimiser escape flat regions in the loss landscape. |
| `ModelCheckpoint` | Saves the best model weights to disk whenever `val_accuracy` improves. |

Learning curves (loss and accuracy vs epoch) are plotted after training — the gap between train and val curves indicates overfitting.

In [ ]:
"""
Compile and train the 1D CNN model.

READING THE LEARNING CURVES
-----------------------------
After this cell you will see two plots:

LEFT  — Loss vs epoch:
  • Both curves should decrease overall.
  • If train loss keeps falling but val loss starts RISING, the model is
    overfitting (memorising training samples rather than learning to generalise).
  • EarlyStopping will catch this and restore the best checkpoint.

RIGHT — Accuracy vs epoch:
  • Both curves should increase overall.
  • The gap between train_acc and val_acc is the overfitting gap.
  • A perfectly fitting model has val_acc close to train_acc.

The vertical dotted line marks the epoch at which val_loss was lowest
(i.e. the weights that EarlyStopping restores).
"""

# ── Callbacks ─────────────────────────────────────────────────────────────────
cb_list = [
    callbacks.EarlyStopping(
        monitor              = 'val_loss',         # watch validation loss
        patience             = EARLY_STOP_PATIENCE, # tolerate this many non-improving epochs
        restore_best_weights = True,               # load best weights at the end
        verbose              = 1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,                            # multiply LR by 0.5 on plateau
        patience = max(3, EARLY_STOP_PATIENCE // 3),
        min_lr   = 1e-6,                           # floor — don't go below this
        verbose  = 1,
    ),
    callbacks.ModelCheckpoint(
        filepath          = MODEL_SAVE_PATH,
        monitor           = 'val_accuracy',        # save when val_accuracy improves
        save_best_only    = True,
        save_weights_only = False,                 # save full model (architecture + weights)
        verbose           = 0,
    ),
]

print(f"Training for up to {EPOCHS} epochs "
      f"(early stopping patience = {EARLY_STOP_PATIENCE} epochs)...")

# ── model.fit — the main training loop ───────────────────────────────────────
# Each epoch:
#   1. Shuffle the training set.
#   2. Split it into batches of BATCH_SIZE.
#   3. For each batch: forward pass → compute loss → backprop → update weights.
#   4. After all batches: compute val_loss and val_accuracy on X_val.
#   5. Callbacks decide whether to stop, reduce LR, or save checkpoint.
history = model.fit(
    X_train, y_train_oh,                     # training data and one-hot labels
    validation_data = (X_val, y_val_oh),     # validation data (no weight updates here)
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = cb_list,
    verbose         = 1,                     # print progress per epoch
)

# ── Learning curves ───────────────────────────────────────────────────────────
hist       = history.history              # dict of lists: loss, val_loss, accuracy, ...
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Best epoch = epoch with lowest validation loss (what EarlyStopping chose)
best_epoch = np.argmin(hist['val_loss']) + 1  # +1 because epochs are 1-indexed

# ── Loss subplot ──────────────────────────────────────────────────────────────
axes[0].plot(epochs_ran, hist['loss'],     label='Train loss', color='#20808D', linewidth=2)
axes[0].plot(epochs_ran, hist['val_loss'], label='Val loss',   color='#A84B2F', linewidth=2, linestyle='--')
axes[0].axvline(best_epoch, color='gray', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[0].set_title('Cross-entropy loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9)
axes[0].spines[['top', 'right']].set_visible(False)

# ── Accuracy subplot ──────────────────────────────────────────────────────────
axes[1].plot(epochs_ran, hist['accuracy'],     label='Train acc', color='#20808D', linewidth=2)
axes[1].plot(epochs_ran, hist['val_accuracy'], label='Val acc',   color='#A84B2F', linewidth=2, linestyle='--')
axes[1].axvline(best_epoch, color='gray', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[1].set_title('Classification accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.02)
axes[1].legend(fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)

fig.suptitle('1D CNN — Training curves', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/02_cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(hist['val_accuracy'])
print(f"\nBest validation accuracy : {best_val_acc:.4f}  (epoch {best_epoch})")
print(f"Training curves saved   → results/02_cnn_training_curves.png")


---

## Cell 9 — Evaluation on the Test Set

The model is evaluated on the held-out test set (never seen during training or validation).  
Outputs:
- **Overall test accuracy / loss** — a single-number summary.
- **Classification report** — per-class precision, recall, F1-score (metrics you know from COMP3308).
- **Confusion matrix** — shows which gestures are confused with which; a perfect classifier has a diagonal matrix.

In [ ]:
"""
Evaluate the trained model on the held-out test set.

METRICS RECAP (from COMP3308)
------------------------------
Precision (per class C):
    TP_C / (TP_C + FP_C)
    "Of all samples predicted as C, what fraction really are C?"

Recall (per class C):
    TP_C / (TP_C + FN_C)
    "Of all samples that truly are C, what fraction did we predict as C?"

F1-score:
    2 × (precision × recall) / (precision + recall)
    Harmonic mean of precision and recall — useful when classes are imbalanced.

Confusion Matrix:
    Rows = true labels, Columns = predicted labels.
    Diagonal entries = correct predictions.
    Off-diagonal entries = misclassifications.
    Example: cm[0,2] = 5 means 5 samples of class 0 were predicted as class 2.

We show TWO versions of the matrix:
  Raw count   — absolute number of samples in each cell.
  Normalised  — each row divided by its total (= recall per class).
"""

# ── Load best checkpoint from disk ───────────────────────────────────────────
# EarlyStopping already restored the best in-memory weights, but loading the
# saved .keras file confirms the disk save is valid and reproduces results.
try:
    best_model = keras.models.load_model(MODEL_SAVE_PATH)
    print(f"Loaded best model from: {MODEL_SAVE_PATH}")
except Exception as e:
    print(f"Could not load checkpoint ({e}), using current in-memory model.")
    best_model = model

# ── Generate predictions ──────────────────────────────────────────────────────
# predict() returns a (N_test, N_classes) probability matrix.
# argmax selects the class with the highest probability — this is the predicted label.
y_pred_prob = best_model.predict(X_test, verbose=0)   # shape (N_test, 4)
y_pred      = np.argmax(y_pred_prob, axis=1)           # shape (N_test,)  integer labels

test_loss, test_acc = best_model.evaluate(X_test, y_test_oh, verbose=0)
print(f"\nTest loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}  ({test_acc * 100:.2f}%)")

# ── Classification report ─────────────────────────────────────────────────────
print("\n" + "─" * 62)
print("Classification report (test set)")
print("─" * 62)
print(classification_report(
    y_test, y_pred,
    target_names = CLASS_NAMES,
    digits       = 4,
))

# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm      = confusion_matrix(y_test, y_pred)                         # raw counts
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)      # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(max(10, N_CLASSES * 1.3),
                                         max(8,  N_CLASSES * 1.0)))

for ax, data, fmt, title in zip(
    axes,
    [cm,      cm_norm],
    ['d',     '.2f'],
    ['Raw count confusion matrix', 'Normalised confusion matrix (row = recall)'],
):
    sns.heatmap(
        data,
        annot       = True,
        fmt         = fmt,
        cmap        = 'Blues',
        xticklabels = CLASS_NAMES,
        yticklabels = CLASS_NAMES,
        linewidths  = 0.4,
        linecolor   = '#cccccc',
        ax          = ax,
        annot_kws   = {'size': max(6, 10 - N_CLASSES // 3)},
    )
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=8)

fig.suptitle('1D CNN — Confusion matrices (test set)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/02_cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved → results/02_cnn_confusion_matrix.png")


---

## Cell 10 — Save Model and Results

Serialises all results (config snapshot, dataset info, training metrics, test performance, confusion matrix) to a JSON file.  
The model is also saved in Keras's native `.keras` format for later inference or transfer learning.

In [ ]:
"""
Save the best model and a complete results JSON to disk.

WHY SAVE A RESULTS JSON?
------------------------
The 00_Results_Comparison.ipynb notebook reads the JSON files produced by ALL
model notebooks (01_LSTM, 02_CNN, 03_CNN_LSTM, etc.) and generates a comparison
table.  Every key in `results` below corresponds to a column in that table.

The JSON also captures the full config at the time of the run, so you can
reproduce any experiment by restoring these hyperparameter values.
"""

from sklearn.metrics import classification_report as sk_cr

# Build a complete, serialisable results dictionary
cr_dict = sk_cr(
    y_test, y_pred,
    target_names = CLASS_NAMES,
    output_dict  = True,   # return dict instead of formatted string
)

results = {
    "model"     : "1D_CNN_Gesture",
    "timestamp" : datetime.datetime.now().isoformat(),

    # ── Configuration snapshot ────────────────────────────────────────────────
    # Record every hyperparameter so the result can be reproduced exactly.
    "config": {
        "filter_type"        : FILTER_TYPE,
        "target_len"         : TARGET_LEN,
        "normalization"      : NORMALIZATION,
        "per_sample_norm"    : PER_SAMPLE_NORM,
        "conv_blocks"        : CONV_BLOCKS,
        "use_batch_norm"     : USE_BATCH_NORM,
        "use_max_norm"       : USE_MAX_POOL,
        "pool_size"          : POOL_SIZE,
        "dropout_rate"       : DROPOUT_RATE,
        "dense_units"        : DENSE_UNITS,
        "global_pool"        : GLOBAL_POOL,
        "epochs"             : EPOCHS,
        "batch_size"         : BATCH_SIZE,
        "learning_rate"      : LEARNING_RATE,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    },

    # ── Dataset info ──────────────────────────────────────────────────────────
    "dataset": {
        "n_samples"    : int(X.shape[0]),
        "n_timesteps"  : int(X.shape[1]),
        "n_features"   : int(N_FEATURES),
        "n_classes"    : int(N_CLASSES),
        "class_names"  : CLASS_NAMES,
        "feature_cols" : FEATURE_COLS_USED,
        "train_size"   : int(X_train.shape[0]),
        "val_size"     : int(X_val.shape[0]),
        "test_size"    : int(X_test.shape[0]),
    },

    # ── Training outcomes ─────────────────────────────────────────────────────
    "training": {
        "epochs_ran"          : len(hist['loss']),
        "best_epoch"          : int(best_epoch),
        "best_val_accuracy"   : float(best_val_acc),
        "best_val_loss"       : float(min(hist['val_loss'])),
        "final_train_accuracy": float(hist['accuracy'][-1]),
        "final_train_loss"    : float(hist['loss'][-1]),
    },

    # ── Test-set performance ──────────────────────────────────────────────────
    "evaluation": {
        "test_loss"             : float(test_loss),
        "test_accuracy"         : float(test_acc),
        "classification_report" : cr_dict,
        "confusion_matrix"      : cm.tolist(),   # .tolist() converts numpy → plain Python list
    },
}

# Serialise to JSON
with open(RESULTS_SAVE_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results JSON saved → {RESULTS_SAVE_PATH}")

# Save the best model (already saved by ModelCheckpoint, but saving again is safe)
best_model.save(MODEL_SAVE_PATH)
print(f"Model saved        → {MODEL_SAVE_PATH}")

# Summary printout
print("\n" + "═" * 52)
print("FINAL RESULTS — 1D CNN")
print("═" * 52)
print(f"  Test accuracy   : {test_acc:.4f}  ({test_acc * 100:.2f} %)")
print(f"  Test loss       : {test_loss:.4f}")
print(f"  Macro F1        : {cr_dict['macro avg']['f1-score']:.4f}")
print(f"  Weighted F1     : {cr_dict['weighted avg']['f1-score']:.4f}")
print("═" * 52)


---

## Cell 11 — Feature Importance via Gradient Saliency Maps *(optional)*

Saliency maps reveal **which time steps and which sensor channels** the network attended to most when making a prediction.

### How it works

After training, we pick one test sample per class and use `tf.GradientTape` to compute:

\[ \text{saliency}(t, c) = \left| \frac{\partial \, \hat{y}_{\text{true}}}{\partial x_{t,c}} \right| \]

— the absolute value of the gradient of the **predicted score for the true class** with respect to each input value.

**Intuition:** If changing a sensor reading at a particular time step would change the prediction score a lot (large gradient), that reading must be important for the decision.

This is a **first-order linear approximation** — valid near the current input point.  For richer attribution consider Integrated Gradients or SHAP DeepExplainer.

### Two plot types produced

1. **Temporal saliency** (one line per class) — averaged over all channels, shows *when* during the gesture the model pays attention.
2. **2D saliency heatmap** (one class) — shows *which sensor at which timestep* was most important.

In [ ]:
"""
Gradient saliency analysis — visualise which parts of the input signal drive
the model's prediction for each gesture class.

WHAT IS A GRADIENT?
--------------------
You learned backpropagation in COMP3308: gradients flow backwards through the
network from the loss to the weights, telling us how much each weight affected
the loss.

Here we apply the same idea differently: instead of computing d(loss)/d(weights),
we compute d(predicted_score)/d(input_values).

  Large |gradient| at (timestep t, sensor channel c):
      "If I changed sensor c's reading at time t slightly, the prediction score
       for the true class would change a lot."  → This input dimension is important.

  Small |gradient| at (timestep t, sensor channel c):
      "Changing this value hardly affects the prediction."  → Less important.

PRACTICAL NOTE
--------------
This is a LOCAL explanation — it tells you why the network made THIS prediction
for THIS specific sample.  It does not tell you about the global importance of
a feature across the whole dataset.
"""
SALIENCY_MAX_CLASSES = min(N_CLASSES, 6)   # limit to 6 panels for readability

# ── Step 1: Pick one representative test sample per class ─────────────────────
# We prefer correctly-classified samples because their saliency maps reflect
# what the model genuinely learned.  Misclassified samples may show confused
# activations that are harder to interpret.
sample_indices = []
sample_labels  = []

for cls in range(SALIENCY_MAX_CLASSES):
    idx_candidates = np.where(y_test == cls)[0]   # all test indices for this class
    if len(idx_candidates) == 0:
        continue
    # Among candidates, prefer those the model got right
    correct = idx_candidates[y_pred[idx_candidates] == cls]
    chosen  = correct[0] if len(correct) > 0 else idx_candidates[0]
    sample_indices.append(chosen)
    sample_labels.append(cls)

# ── Step 2: Compute gradients using tf.GradientTape ───────────────────────────
def compute_saliency(model, X_sample, true_class):
    """
    Compute the input-gradient saliency map for ONE sample.

    tf.GradientTape records all operations on watched tensors during the
    forward pass.  Calling tape.gradient() then computes the gradient of the
    specified scalar (predicted score for true_class) with respect to the
    watched tensor (the input).

    Parameters
    ----------
    model      : trained Keras model
    X_sample   : np.ndarray shape (T, C) — one test trial
    true_class : int — the ground-truth class index

    Returns
    -------
    saliency : np.ndarray shape (T, C) — absolute gradient magnitudes
    """
    # Add batch dimension: (T, C) → (1, T, C)
    # tf.constant makes the tensor non-trainable, but tape.watch() overrides that.
    x_tensor = tf.constant(X_sample[np.newaxis, :, :], dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)           # tell the tape to track gradients wrt input
        preds = model(x_tensor, training=False)   # forward pass (batch norm in inference mode)
        score = preds[0, true_class]              # scalar: predicted probability for true class

    # Differentiate score with respect to every input element
    grads    = tape.gradient(score, x_tensor)    # shape (1, T, C)
    saliency = tf.abs(grads[0]).numpy()           # shape (T, C)  — take absolute value
    return saliency

# Run saliency computation for each selected sample
saliency_maps = []
for idx, cls in zip(sample_indices, sample_labels):
    sal = compute_saliency(best_model, X_test[idx], cls)
    saliency_maps.append(sal)

print(f"Computed saliency maps for {len(saliency_maps)} class(es).")

# ── Plot 1: Temporal saliency (mean over channels) ────────────────────────────
# For each sample: average the absolute gradient over the channel dimension
# → gives a 1D curve showing WHEN in the gesture the model paid most attention.
n_panels = len(saliency_maps)
ncols    = min(n_panels, 3)
nrows    = (n_panels + ncols - 1) // ncols
time_axis = np.arange(TARGET_LEN) / FS_HZ * 1000   # convert timesteps → milliseconds

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 5, nrows * 2.5),
                          squeeze=False)

for panel_idx, (sal, cls) in enumerate(zip(saliency_maps, sample_labels)):
    ax = axes[panel_idx // ncols][panel_idx % ncols]

    # Average over channel dimension: shape (T, C) → (T,)
    temporal_sal = sal.mean(axis=1)

    # Normalise to [0, 1] for easy visual comparison across classes
    temporal_sal = (temporal_sal - temporal_sal.min()) /                    (temporal_sal.max() - temporal_sal.min() + 1e-9)

    ax.fill_between(time_axis, temporal_sal, alpha=0.4, color='#20808D')
    ax.plot(time_axis, temporal_sal, color='#1B474D', linewidth=1.2)

    # Mark the top-3 most salient time points with red dots
    top_t = np.argsort(temporal_sal)[-3:]
    ax.scatter(time_axis[top_t], temporal_sal[top_t],
               color='#A84B2F', s=40, zorder=5)

    # Show whether the model predicted this sample correctly
    pred_cls    = y_pred[sample_indices[panel_idx]]
    correct_str = '✓' if pred_cls == cls else f'✗→{CLASS_NAMES[pred_cls]}'

    ax.set_title(f"{CLASS_NAMES[cls]} {correct_str}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (ms)', fontsize=8)
    ax.set_ylabel('Normalised saliency', fontsize=8)
    ax.set_ylim(-0.05, 1.1)
    ax.spines[['top', 'right']].set_visible(False)

# Hide any unused subplot panels
for extra in range(n_panels, nrows * ncols):
    axes[extra // ncols][extra % ncols].set_visible(False)

fig.suptitle('1D CNN — Temporal saliency (mean |gradient| over all sensor channels)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/02_cnn_temporal_saliency.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: 2D saliency heatmap (time × sensor channel) for the first class ───
# Shows the full (T, C) gradient matrix as a colour map.
# Bright cells = the model was sensitive to that specific sensor at that moment.
if len(saliency_maps) > 0:
    sal0     = saliency_maps[0]     # (T, C)
    cls0     = sample_labels[0]
    n_show_c = min(N_FEATURES, 30)  # cap at 30 channels to keep the plot legible

    # Rank channels by their mean saliency; show only the most informative ones
    channel_mean_sal = sal0.mean(axis=0)                          # (C,)
    top_channel_idx  = np.argsort(channel_mean_sal)[::-1][:n_show_c]
    top_channel_names = [FEATURE_COLS_USED[i] for i in top_channel_idx]

    sal_subset = sal0[:, top_channel_idx].T   # shape (n_show_c, T) for imshow

    fig, ax = plt.subplots(figsize=(12, max(4, n_show_c * 0.35)))
    im = ax.imshow(
        sal_subset,
        aspect = 'auto',
        origin = 'upper',
        cmap   = 'YlOrRd',          # yellow → orange → red (low → high saliency)
        extent = [0, TARGET_LEN / FS_HZ * 1000, n_show_c, 0],
    )
    ax.set_yticks(np.arange(n_show_c) + 0.5)
    ax.set_yticklabels(top_channel_names, fontsize=7)
    ax.set_xlabel('Time (ms)', fontsize=10)
    ax.set_title(
        f'2D saliency heatmap — class: {CLASS_NAMES[cls0]}  '
        f'(top {n_show_c} channels by mean |gradient|)',
        fontsize=11, fontweight='bold',
    )
    cbar = fig.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('|∂score / ∂input|', fontsize=9)
    plt.tight_layout()
    plt.savefig('results/02_cnn_channel_saliency.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Saliency analysis complete.  Figures saved:")
print("  results/02_cnn_temporal_saliency.png")
print("  results/02_cnn_channel_saliency.png")
